In [2]:
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast

model_name = "reverse-gpt2-0.35B-fineweb-10BT-ctx-1024-chat-v15"

MODEL_DIR = "/home/wyf/orcd/pool/reverse-llm/models"
TOKENIZER_DIR = "/home/wyf/orcd/pool/reverse-llm/tokenizers"
DATA_DIR = "/home/wyf/orcd/pool/reverse-llm/data"

context_length = 1024

model = GPT2LMHeadModel.from_pretrained(f"{MODEL_DIR}/{model_name}/checkpoint-550")

In [3]:
USER_ROLE_NAME = "user"[::-1]
ASSISTANT_ROLE_NAME = "assistant"[::-1]

In [4]:
tokenizer = PreTrainedTokenizerFast.from_pretrained(f"{TOKENIZER_DIR}/fineweb_bpe_200k")
tokenizer.add_special_tokens({ "additional_special_tokens": ["<im_start>", "<im_end>"] })

tokenizer.chat_template = """{% for message in messages -%}
<im_start>{{ message['role'] }}
{{ message['content'] }}<im_end>
{%- endfor -%}
{% if add_generation_prompt and messages[-1]['role'] != 'assistant' -%}
<im_start>tnatsissa{{ '\n' }}
{%- endif %}"""

In [5]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    # max_new_tokens=128,
    do_sample=True,
    temperature=1.0,
    num_return_sequences=1,
    # pad_token_id=tokenizer.eos_token_id,
    # eos_token_id=tokenizer.eos_token_id,
    # bos_token_id=tokenizer.bos_token_id,
    clean_up_tokenization_spaces=False,
)

Device set to use cuda:0


In [6]:
tokenizer.special_tokens_map

{'bos_token': '<s>',
 'eos_token': '</s>',
 'unk_token': '<unk>',
 'pad_token': '<pad>',
 'mask_token': '<mask>',
 'additional_special_tokens': ['<im_start>', '<im_end>']}

In [6]:
from datasets import Dataset
# tokenized_valid = Dataset.load_from_disk(f"{DATA_DIR}/alpaca/tokenized_{context_length}_valid")
tokenized_valid = Dataset.load_from_disk(f"{DATA_DIR}/ultrachat/tokenized_{context_length}_valid")

print(tokenizer.decode(tokenized_valid[199]['input_ids'])[::-1])

>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<>dap<

In [36]:
from transformers import pipeline

pipe = pipeline(
   "text-generation",
   model=model,
   tokenizer=tokenizer,
   clean_up_tokenization_spaces=False,
)

query = """
How do you treat a jellyfish stab?
""".strip()

messages = [
   {"role": USER_ROLE_NAME, "content": query[::-1]},
]
# For generation, apply template with add_generation_prompt=True
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(f"Tokenized prompt: {prompt}")

outputs = pipe(
   prompt,
   num_return_sequences=1,
   # eos_token_id=tokenizer.eos_token_id,
   # max_new_tokens=128,
   eos_token_id=52001,
   pad_token_id=tokenizer.pad_token_id,
   repetition_penalty=1.1,
   temperature=0.7,
)
out_text = outputs[0]["generated_text"]
response = out_text.split("<im_start>tnatsissa")[1]

print(f"\n=== QUERY ===\n{query}\n")
print(f"=== RESPONSE ===\n{response[::-1]}")

Device set to use cuda:0


Tokenized prompt: <im_start>resu
?bats hsifyllej a taert uoy od woH<im_end><im_start>tnatsissa


=== QUERY ===
How do you treat a jellyfish stab?

=== RESPONSE ===
Crab depends on its intended use. Crab is very effective and can be applied directly on the skin of the fish being used. It can be done in a few different ways. The most common method is to remove the long, slightly tough outer shell from the jellyfish and insert it into the fish using a hook inserted into the thickest part of the fish. If you must try something that is not to your liking, contact your healthcare professional and ask for help. There's no right or wrong way to do it!

